# 05 — "What is your name?": what the engine did, with the open items closed
Questions left open by the 2026-10-17 trace: (1) why is `holcus` first for every prompt, (2) why does its slot clear `J_floor`, (3) was the identity text learned into this store.

In [1]:
import os, sys, json, math, collections, re, hashlib
import numpy as np
sys.path.insert(0, os.path.abspath("../tools"))
import ptol_bin, monad_py as mp
ROOT  = os.path.abspath("..")
STORE = os.path.join(ROOT, "dirty", "monad_wordnet_v4_20260519.bin")
CBIN  = os.path.join(ROOT, "build", "PtolC_eb28ce8", "ptolemy")
SPACES = "[\u200b\u2009\u2008\u2007\u2006\u2005\u2004\u2003]"   # the eight Fermat separators

In [2]:
m = mp.Monad.from_bin(STORE); H = mp.word_coords("holcus")[0]
print("holcus ->", mp.word_coords("holcus"), " seed =", (mp.word_coords("holcus")[1]-mp.D_STAR)/(mp.OMEGA_ZS-mp.D_STAR), "= 8074/8192 =", 8074/8192)

holcus -> (24639, 0.5625142041015625)  seed = 0.9855957031250001 = 8074/8192 = 0.985595703125


## (2) Why the slot clears `J_floor`: it is a hub
The slot has 18 268 A-neighbours (the median slot has 150) and is the 5th best-connected slot of 25 000. It shares an edge with all four query slots, so it receives
weight in the A-passes whatever the prompt.

In [3]:
deg = np.bincount(np.concatenate([m.Ai, m.Aj]), minlength=25000)
print("degree of holcus slot", deg[H], " rank", int((deg>deg[H]).sum())+1, " median degree", int(np.median(deg)), " top-5 degrees", np.sort(deg)[::-1][:5])
q = [a[0] for a in m.hear_raw("what is your name")]
print("direct edges to the query slots:", [(int(i),int(j),round(float(w),4)) for i,j,w in zip(m.Ai,m.Aj,m.Aw) if (i==H and j in q) or (j==H and i in q)])

degree of holcus slot 18268  rank 5  median degree 150  top-5 degrees [22987 20024 19113 18935 18268]


direct edges to the query slots: [(15471, 24639, 0.1688), (16194, 24639, 0.3559), (1894, 24639, 0.6468), (18893, 24639, 0.9547)]


## The slot is the slot of the word *in*
Which words address to z#24639? Every WordNet lemma and every function word is tested; two do. The slot's displayed word is whichever has the higher `E`.

In [4]:
from nltk.corpus import wordnet as wn
lem = sorted({l.lower() for s in wn.all_synsets() for l in s.lemma_names() if l.isalpha() and l.isascii()})
cand = sorted({(mp.word_coords(w)[1], w) for w in lem + ["in","of","the","and","to","a","is"] if mp.token_accept(w) and mp.word_coords(w)[0]==H}, reverse=True)
for E, w in cand: print(f"{w:8} E = {E!r}")
print("E difference:", cand[0][0]-cand[1][0])

holcus   E = 0.5625142041015625
in       E = 0.5625130258798404
E difference: 1.1782217220757119e-06


`holcus` beats `in` by about 1·10⁻⁶ in `E` and takes the label; `in` is one of the most frequent words in English, so its slot is a hub. The five best-connected slots are all slots of function words:

In [5]:
order = np.argsort(-deg)[:8]
fw = ["the","and","of","to","in","a","is","that","it","for","as","with","was","by","on","at","be"]
for i in order:
    print(f"z#{int(i):<6} degree {int(deg[i]):6d}  displayed {m.vocab.get(int(i),{}).get('word')!r:10}  function words addressing here: {[w for w in fw if mp.word_coords(w)[0]==int(i)]}")

z#0      degree  22987  displayed 'seemy'     function words addressing here: []
z#841    degree  20024  displayed 'the'       function words addressing here: ['the']
z#7919   degree  19113  displayed 'and'       function words addressing here: ['and']
z#8017   degree  18935  displayed 'qgt'       function words addressing here: ['of']
z#24639  degree  18268  displayed 'holcus'    function words addressing here: ['in']
z#11228  degree  17877  displayed 'to'        function words addressing here: ['to']
z#1894   degree  15340  displayed 'nanti'     function words addressing here: ['is']
z#24362  degree  14924  displayed 'mcany'     function words addressing here: ['that']


## (1) Why it is first: the selector, not the prompt
The first word is the slot, among those with `J > J_floor` and an open gate, whose stored `E` is closest to `target_E = (ΣV/(ΣV+ΣT))·Ω`. Because `ΣT ≪ ΣV`, `target_E ≈ Ω = 0.5671`
for any short prompt. `holcus` has `E = 0.5625`, the closest available.

First word over many prompts (random 1–6 word prompts drawn from the store's own words, fresh state each time):

In [6]:
import random; random.seed(7)
W = [v["word"] for v in m.vocab.values() if mp.token_accept(v["word"])]
first = collections.Counter(); lens = collections.Counter()
for t in range(150):
    q = " ".join(random.sample(W, random.randint(1,6)))
    mm = mp.Monad.from_bin(STORE) if t == 0 else mm
    mm.age = [0]*25000; mm.affect = 0.0; mm.last_pointer = 0
    ws = re.split(SPACES, mm.speak_wick(q, 50)); first[ws[0]] += 1; lens[len(ws)] += 1
print(first.most_common(6)); print("reply lengths:", dict(lens))

[('holcus', 92), ('yack', 14), ('aiao', 4), ('', 2), ('auvrn', 1), ('weru', 1)]
reply lengths: {4: 140, 2: 4, 1: 4, 3: 2}


`holcus` is first in 92 of 150 random prompts and in all 8 of the standard prompts. Others (`yack`, …) win when `holcus`'s gate or `J` fails for that prompt, or when another slot's `E`
is nearer `target_E`. Reply length is 4 in 140 of 150; shorter replies occur when fewer than 3 walk candidates pass the gates.

## (3) Was the identity text learned into this store?
`PTOLEMY_IDENTITY` (`monad.c` lines 97–139) is learned only with `-i`. Parse it out of the C source, learn it from ground state, and ask whether each of its edges exists in the store with at least that weight.

In [7]:
src = open(os.path.join(ROOT,"src","PtolC_eb28ce8","monad.c")).read()
body = src.split("static const char *PTOLEMY_IDENTITY =")[1].split(";\n")[0]
ident = "".join(re.findall(r'"((?:[^"\\]|\\.)*)"', body)).encode().decode("unicode_escape")
print(len(ident), "chars:", ident[:120], "...")
g = mp.Monad(); g.learn(ident); IA = g._A_dict()
store = {(int(i),int(j)): float(w) for i,j,w in zip(m.Ai,m.Aj,m.Aw)}
present = [k for k in IA if k in store]; geq = [k for k in present if store[k] >= IA[k]*(1-1e-9)]
print("identity edges:", len(IA), " present in the store:", len(present), " with weight >= identity's own contribution:", len(geq))

1881 chars: I am Ptolemy. I am named after Ptolemy the Second, Philadelphos, who commissioned the Septuagint. Seventy two scholars t ...


identity edges: 646  present in the store: 646  with weight >= identity's own contribution: 646


If the store had been built without `-i`, only chance overlap would appear. Edge density of the store is 6 825 748 / (25 000·24 999/2) = 2.18 %, so about 14 of the 646 would be expected by chance; all 646 are present.

In [8]:
rare = [k for k in IA if IA[k] > 0 and k not in store]
print("identity edges absent from the store:", len(rare))
for w in ["ptolemy","philadelphos","septuagint","noether","riemann","prime","zero","holcus"]:
    i,E = mp.word_coords(w); print(f"{w:13} z#{i:<6} slot displays {m.vocab.get(i,{}).get('word')!r}  degree {int(deg[i])}")

identity edges absent from the store: 0
ptolemy       z#10156  slot displays 'sift'  degree 11109
philadelphos  z#0      slot displays 'seemy'  degree 22987
septuagint    z#0      slot displays 'seemy'  degree 22987
noether       z#19921  slot displays 'jenna'  degree 10515
riemann       z#14453  slot displays 'haf'  degree 10080
prime         z#7148   slot displays 'ehttp'  degree 1600
zero          z#2139   slot displays 'sarto'  degree 1922
holcus        z#24639  slot displays 'holcus'  degree 18268


## What this establishes and what it does not
- The reply is produced entirely by the stages in notebook 04; the port reproduces it exactly.
- `holcus` first = the label of the slot where the word *in* lives (a 1·10⁻⁶ difference in `E` between the two words that address there) + that slot being a hub + the selector's `E`-nearest rule + an `E` close to Ω. None of those four reads the prompt.
- The identity text was learned into this store: all 646 edges it would create are present, each at least as heavy as the identity's own contribution (chance expectation at the store's edge density is ≈ 14).
- Words 2–4 come from the golden-angle walk from the pointer through slots with `J > J_floor`; their `J` comes from the A-matrix learned from text.
- Whether this *is* self-understanding is not a code question; the code shows the mechanism that produced the sequence.